# Preparing pre-match World Cup data

Two analyses predict signed goal difference and team goals. There are 104 matches and 208 team appearances, not a random sample of all football. The preparation uses real previous internationals for the pre-match variables.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'analysis.py').exists(): ROOT=ROOT/'Assignment3'
assert (ROOT/'analysis.py').exists()
sys.path.insert(0,str(ROOT.resolve()))
from analysis import prepare_data, run_task, TASK1_X, TASK2_X


## 1. Sources

TSDL supplies 103 fixtures and the recorded FIFA/FBref bronze result adds one. A pinned Mart Jürisoo international-results snapshot corroborates all 104 scores and supplies pre-tournament history. This is a third-party compilation, not official FIFA data. Scores include extra time but exclude shootout kicks. Raw files, source README, CC0 license, commit, SHA-256 and download times are preserved. A post-event download is not proof of a pre-kickoff published snapshot.

In [2]:
print((ROOT/'data/raw/sources.json').read_text())
print((ROOT/'data/raw/history_sources.json').read_text())

[
  {
    "file": "data/raw/tsdl_fixtures.csv",
    "url": "https://docs.google.com/spreadsheets/d/e/2PACX-1vSWZFlaUHTBK09v4I1Kv7ZQ0ophhlpsCr7VPFW5dkbdG0Zpl8mRkXrTZezZMr1Ia9V9cpwmq7BKPQ03/pub?gid=995472238&single=true&output=csv",
    "source_page": "https://www.thestatsdontlie.com/football/world-cup-2026/",
    "downloaded_utc": "2026-10-03T06:47:47.171381+00:00",
    "sha256": "5b3420416acaf1cd65b8beec80eef4f06e99ead9c89135a000ed38be8609cb1b"
  },
  {
    "file": "data/raw/tsdl_tournament_summary.csv",
    "url": "https://docs.google.com/spreadsheets/d/e/2PACX-1vSWZFlaUHTBK09v4I1Kv7ZQ0ophhlpsCr7VPFW5dkbdG0Zpl8mRkXrTZezZMr1Ia9V9cpwmq7BKPQ03/pub?gid=0&single=true&output=csv",
    "source_page": "https://www.thestatsdontlie.com/football/world-cup-2026/",
    "downloaded_utc": "2026-10-03T06:47:50.876725+00:00",
    "sha256": "7a2c3a719eac9d08d66974b7d3e70f00d799b1a0f6595e048c44f594eb19b169"
  }
]
[
  {
    "file": "data/raw/international_results.csv",
    "url": "https://raw.githubuserc

## 2. Clean and rebuild

Parse dates, remove only the shootout marker from scores, check duplicates/counts, and make two team rows per match. Four explicit team-name aliases connect sources. A/B follows fixture order, not winner/loser or necessarily home/away. Identifiers never enter the model.

In [3]:
one,two=prepare_data()
print(one.head().to_string(index=False))
print(two.head().to_string(index=False))
print((ROOT/'results/data_audit.json').read_text())

match_id       date split      team_a               team_b  goal_difference  a_pre_gf  a_pre_ga  a_pre_win_rate  a_pre_clean_sheet_rate  b_pre_gf  b_pre_ga  b_pre_win_rate  b_pre_clean_sheet_rate
    M001 2026-06-11 train      Mexico         South Africa                2       1.8       0.4             0.6                     0.6       0.8       1.0             0.2                     0.4
    M002 2026-06-12 train      Canada Bosnia & Herzegovina                0       1.4       0.6             0.4                     0.6       0.8       0.8             0.0                     0.2
    M003 2026-06-12 train South Korea       Czech Republic                1       1.4       1.0             0.6                     0.6       3.4       1.2             0.6                     0.2
    M004 2026-06-13 train      Brazil              Morocco                0       2.6       1.4             0.6                     0.0       2.6       0.6             0.6                     0.4
    M005 2026-06-13 

## 3. Eight predictors

Four statistics for each side: last-five mean goals scored, mean goals conceded, win proportion and clean-sheet proportion. Four × two = eight predictors. Clean sheet means zero goals conceded. A shootout win after a drawn score counts as a draw here. Five matches is a fixed, simple recency choice, not an optimised window. All rows have five genuine earlier games; there is no imputation.

Use internationals from 2024 to 10 June 2026 plus earlier completed tournament matches. Exclude current, future and same-day games. `feature_history_lineage.csv` records the five contributing matches for every team appearance.

In [4]:
lineage=pd.read_csv(ROOT/'data/processed/feature_history_lineage.csv')
print(lineage[(lineage.team=='Mexico')&(lineage.match_id=='M001')].to_string(index=False))
from prematch import ALIASES
print(ALIASES)

match_id   team forecast_date  history_order history_date history_opponent  gf  ga                    source  source_row history_id
    M001 Mexico    2026-06-11              1   2026-03-28         Portugal   0   0 international_results.csv     49240.0     H49240
    M001 Mexico    2026-06-11              2   2026-03-31          Belgium   1   1 international_results.csv     49292.0     H49292
    M001 Mexico    2026-06-11              3   2026-05-22            Ghana   2   0 international_results.csv     49324.0     H49324
    M001 Mexico    2026-06-11              4   2026-05-30        Australia   1   0 international_results.csv     49336.0     H49336
    M001 Mexico    2026-06-11              5   2026-06-04           Serbia   5   1 international_results.csv     49384.0     H49384
{'United States': 'USA', 'Curaçao': 'Curacao', 'DR Congo': 'D.R. Congo', 'Bosnia and Herzegovina': 'Bosnia & Herzegovina'}


## 4. Source disagreement

All scores agree, but 35 fixture dates differ by one day. The cause is not conclusively verified. TSDL dates consistently define within-tournament chronology and splits; the other source supplies only pre-tournament histories. Exact kickoff time/timezone is unavailable. Historical outcomes were knowable before the match, but post-event corrections to this downloaded snapshot cannot be ruled out.

In [5]:
print(pd.read_csv(ROOT/'data/processed/source_reconciliation_104.csv').date_offset_days.value_counts().to_string())

date_offset_days
 0    69
-1    35


## 5. Evaluation

Training: through 25 June (56 matches); validation: 26 June–1 July (26); later evaluation: 2–19 July (22). Both team rows remain together. Earlier completed evaluation fixtures update later histories, but coefficients and transformations are not refitted. This represents sequential forecasting, not a pre-tournament forecast.

The old version already inspected these outcomes. This revision is retrospective, NOT untouched external validation. Repeated teams and paired rows also limit independence.

In [6]:
import inspect
from prematch import build_features
print(inspect.getsource(build_features))

def build_features(matches, history, window=5):
    """History source only before tournament; within-tournament history from fixtures.

    A historical snapshot downloaded after the event is a retrospective proxy,
    not proof that this particular CSV version was published before kickoff.
    """
    historical = history[(history.date >= '2024-01-01') & (history.date < '2026-06-11')]
    past = []
    for r in historical.itertuples():
        for team, opp, gf, ga in [(r.home_team,r.away_team,r.home_score,r.away_score),
                                  (r.away_team,r.home_team,r.away_score,r.home_score)]:
            past.append(dict(team=team, opponent=opp, date=r.date, gf=gf, ga=ga,
                source='international_results.csv', source_row=r.source_csv_row,
                history_id=f'H{r.source_csv_row}', source_match_id=''))
    for r in matches.itertuples():
        for team, opp, gf, ga in [(r.team_a,r.team_b,r.goals_a,r.goals_b),
                                  (r.tea